In [15]:
import time
import pandas as pd
from google_play_scraper import Sort, reviews
from tqdm import tqdm

app_categories = {
    "Shopping": {
        "Flipkart": "com.flipkart.android",
        "Amazon India": "in.amazon.mShop.android.shopping",
        "Myntra": "com.myntra.android",
        "Meesho": "com.meesho.supply",
        "AJIO": "com.ril.ajio",
        "Nykaa": "com.fsn.nykaa",
    },
    
    "Delivery": {
        "Zepto": "com.zeptoconsumerapp",
        "Blinkit": "com.grofers.customerapp",
        "Zomato": "com.application.zomato",
        "Swiggy": "in.swiggy.android",
        "Uber": "com.ubercab",
        "Rapido": "com.rapido.passenger",
    },
    
    "FinTech": {
        "PhonePe": "com.phonepe.app",
        "Google Pay": "com.google.android.apps.nbu.paisa.user",
        "Paytm": "net.one97.paytm",
        "CRED": "com.dreamplug.androidapp",
    },
    
    "Social": {
        "Instagram": "com.instagram.android",
        "WhatsApp": "com.whatsapp",
        "LinkedIn": "com.linkedin.android",
        "YouTube": "com.google.android.youtube",
    },
    
    "Productivity": {
        "Notion": "notion.id",
        "Google Calendar": "com.google.android.calendar",
        "Todoist": "com.todoist",
        "TickTick": "com.ticktick.task",
    },
    
    "Media_Travel": {
        "JioCinema": "com.jio.media.ondemand",
        "Spotify": "com.spotify.music",
        "MakeMyTrip": "com.makemytrip",
    },
}


# 2. CONFIGURATION & PIPELINE EXECUTION
REVIEWS_PER_APP = 4000  # Total Apps ~ 27 * 4000 = ~1,08,000 Reviews
all_scraped_reviews = []

total_apps = sum(len(apps) for apps in app_categories.values())
pbar = tqdm(total=total_apps, desc="Overall Progress")

for category_name, apps_dict in app_categories.items():
    for app_name, package_id in apps_dict.items():
        try:
            # Scrape reviews across all star ratings
            fetched_reviews, _ = reviews(
                package_id,
                lang="en",
                country="in",  # 'in' for India specific app store context
                sort=Sort.MOST_RELEVANT,
                count=REVIEWS_PER_APP,
            )

            # Convert to DataFrame
            df_temp = pd.DataFrame(fetched_reviews)

            # Keep required columns only
            df_temp = df_temp[["content", "score", "at", "thumbsUpCount"]]
            df_temp["app_name"] = app_name
            df_temp["category"] = category_name

            all_scraped_reviews.append(df_temp)

        except Exception as e:
            print(f"\n Warning: Failed to scrape {app_name} ({package_id}): {e}")

        pbar.update(1)
        time.sleep(1)  # Polite delay to prevent rate-limiting

pbar.close()


# 3. CONSOLIDATION, CLEANING & PREPROCESSING
# Combine all lists into a single pandas DataFrame
master_df = pd.concat(all_scraped_reviews, ignore_index=True)

# 1. Drop Null/NaN content entries
master_df.dropna(subset=["content"], inplace=True)
master_df["content"] = master_df["content"].astype(str)

# 2. Remove duplicate reviews
initial_count = len(master_df)
master_df.drop_duplicates(subset=["content"], inplace=True)
print(f" Removed {initial_count - len(master_df)} duplicate reviews.")

# 3. Create Binary Sentiment Label (0 = Negative/Complaint, 1 = Positive)
master_df = master_df[master_df["score"] != 3].copy()
master_df["label"] = (master_df["score"] >= 4).astype(int)

master_df.reset_index(drop=True, inplace=True)

print("\n Reviews Count by Category:")
print(master_df["category"].value_counts())

print("\n Rating Distribution:")
print(master_df["score"].value_counts().sort_index())

master_df=master_df.to_csv("C:/Users/shrut/Desktop/PLAYSTORE_NLP/VS_CODE/main.csv")


Overall Progress: 100%|██████████| 27/27 [02:13<00:00,  4.94s/it]


 Removed 60 duplicate reviews.

 Reviews Count by Category:
category
Shopping        23165
Delivery        22892
FinTech         14847
Social          14456
Productivity    13774
Media_Travel    11168
Name: count, dtype: int64

 Rating Distribution:
score
1    66523
2     9146
4     7265
5    17368
Name: count, dtype: int64


In [55]:
import pandas as pd 

In [47]:
main=pd.read_csv("C:/Users/shrut/Desktop/PLAYSTORE_NLP/VS_CODE/main.csv")

In [48]:
df=main.copy()

In [49]:
import re
def preprocess(text):
    if not isinstance(text, str):
        return ""
    text=re.sub(r'<[^>]+>','',text)                      #html
    text=re.sub(r'https?://\S+|www.\.\S+','',text)       #url
    text=text.encode('ascii','ignore').decode('ascii')   #emoji
    text=text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)           #punctuation
    text = re.sub(r'\s+', ' ', text).strip()             #white spaces

    return text
df["content"]=df["content"].apply(preprocess)


In [50]:
basic_nlp=df.to_csv("C:/Users/shrut/Desktop/PLAYSTORE_NLP/VS_CODE/basic_nlp.csv")

In [56]:
basic_nlp=pd.read_csv("C:/Users/shrut/Desktop/PLAYSTORE_NLP/VS_CODE/basic_nlp.csv")

In [57]:
basic_nlp.isnull().sum()

Unnamed: 0.1     0
Unnamed: 0       0
content          6
score            0
at               0
thumbsUpCount    0
app_name         0
category         0
label            0
dtype: int64

In [60]:
basic_nlp=basic_nlp.dropna()

In [61]:
basic_nlp.isnull().sum()

Unnamed: 0.1     0
Unnamed: 0       0
content          0
score            0
at               0
thumbsUpCount    0
app_name         0
category         0
label            0
dtype: int64

In [62]:
df2=basic_nlp.copy()

In [63]:
from nltk.corpus import stopwords
stop_words=set(stopwords.words('english'))
def remove_stopwords(text):
    if not isinstance(text, str):
        return ""
    return " ".join([word for word in text.split() if word not in stop_words])
df2['content']=df2['content'].apply(remove_stopwords)


In [64]:
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('V'):  # Verbs (e.g., running -> run)
        return 'v'
    elif tag.startswith('J'):  # Adjectives (e.g., better -> good)
        return 'a'
    elif tag.startswith('R'):  # Adverbs (e.g., quickly -> quick)
        return 'r'
    else:
        return 'n'  # Default to Noun

def proper_lemmatizer(text):
    if not isinstance(text, str):
        return ""
    
    words = text.split()
    tagged_words = pos_tag(words)
    clean_words = [lemmatizer.lemmatize(word, pos=get_wordnet_pos(tag)) for word, tag in tagged_words]
    
    return " ".join(clean_words)

df2['content'] = df2['content'].apply(proper_lemmatizer)


In [38]:
after_stopwd_lemm=df2.to_csv("C:/Users/shrut/Desktop/PLAYSTORE_NLP/VS_CODE/after_stopwd_lemm.csv")

In [39]:
after_stopwd_lemm=pd.read_csv("C:/Users/shrut/Desktop/PLAYSTORE_NLP/VS_CODE/after_stopwd_lemm.csv")

In [40]:
df3=after_stopwd_lemm.copy()